# Assemble the Task 3 + Task 4 improvement draft

This notebook starts from the verified `kaggle_t12345_current_v5` file, preserving its Task 1, 2, and 5 predictions. It reads the selected Task 3 model and its linked EDA Test split by their JSON cards, and the Task 4 per-task prediction artifact. It only replaces columns 3 and 4, then verifies the full submission against the current sample. No Test labels or Test fitting are used.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, sys, joblib
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
NAME = "kaggle_t12345_gate_resnet_v6"
BASE_NAME = "kaggle_t12345_current_v5"
base_card_path = SRC / "submission" / f"{BASE_NAME}.json"
base_card = json.loads(base_card_path.read_text(encoding="utf-8"))
base_path = SRC / base_card["submission_path"]
assert hashlib.sha256(base_path.read_bytes()).hexdigest() == base_card["sha256"]
base = pd.read_csv(base_path)
sample = pd.read_csv(SRC / "submission" / "sample_submission.csv")
columns = ["id", "task1", "task2", "task3", "task4", "task5"]
assert base.columns.tolist() == sample.columns.tolist() == columns
assert len(base) == len(sample) == 25_889
assert base.id.tolist() == sample.id.tolist() and sample.id.is_unique
print('Verified base:', BASE_NAME, len(base))

Verified base: kaggle_t12345_current_v5 25889


## Recalculate Task 3 from its saved model and linked Test split

Task 3 has player IDs rather than `ANS` IDs. The unchanged Task 3 Test manifest maps to the sample by row position, as checked in the existing assembly workflow. The model was trained on development rows only.

In [2]:
t3_name = "t3_gate_blend_cv5_v2"
t3_card_path = SRC / "task3" / "save" / f"{t3_name}.json"
t3_card = json.loads(t3_card_path.read_text(encoding="utf-8"))
t3_data = json.loads((SRC / t3_card["data_card_path"]).read_text(encoding="utf-8"))
assert t3_card["data_name"] == t3_data["name"] == "eda_combined_v1"
t3_test_path = SRC / t3_data["splits"]["test"]["path"]
t3_test = pd.read_parquet(t3_test_path) if t3_data["format"] == "parquet" else pd.read_csv(t3_test_path)
assert len(t3_test) == len(sample) and t3_test.player_id.is_unique
t3_model = joblib.load(SRC / t3_card["model_path"])
t3_pred = np.asarray(t3_model.predict(t3_test.drop(columns="player_id")), dtype=float).reshape(-1)
assert len(t3_pred) == len(sample) and np.isfinite(t3_pred).all() and (t3_pred >= 0).all()
t3_eval = json.loads((SRC / "task3" / "save" / f"{t3_name}-evaluation.json").read_text(encoding="utf-8"))
assert t3_eval["metric"] == "MAE_thb"
print('Task 3 holdout MAE THB:', t3_eval['score'])
print('Task 3 prediction summary:')
display(pd.Series(t3_pred).describe(percentiles=[.5,.9,.99]))

Task 3 holdout MAE THB: 2845.0992169223155
Task 3 prediction summary:


count     25889.000000
mean      10559.811020
std       27441.221557
min           0.000000
50%          96.684045
90%       42568.983498
99%      131640.293439
max      251558.766834
dtype: float64

## Read the selected Task 4 per-task prediction artifact

The Task 4 file already links to the saved full-data geometry ResNet18 and uses current `ANS` IDs. Join by ID rather than row position.

In [3]:
t4_name = "t4_resnet18_geometry_full_v2"
t4_part_card_path = SRC / "task4" / "submission" / f"{t4_name}.json"
t4_part_card = json.loads(t4_part_card_path.read_text(encoding="utf-8"))
assert t4_part_card["model_name"] == t4_name and t4_part_card["data_name"] == "eda_1_v2"
t4_part = pd.read_csv(SRC / t4_part_card["submission_path"], sep="|")
assert t4_part.columns.tolist() == ["id", "task4"]
assert len(t4_part) == len(sample) and t4_part.id.is_unique
assert set(t4_part.id) == set(sample.id)
assert set(t4_part.task4) <= set(range(5))
t4_by_id = sample[["id"]].merge(t4_part, on="id", how="left", validate="one_to_one", sort=False)
assert t4_by_id.task4.notna().all()
t4_eval = json.loads((SRC / "task4" / "save" / f"{t4_name}-evaluation.json").read_text(encoding="utf-8"))
assert t4_eval["metric"] == "macro_F1_grouped_cv3"
print('Task 4 grouped three-fold procedure estimate:', t4_eval['score'])
print('Task 4 prediction counts:', t4_by_id.task4.value_counts().sort_index().to_dict())

Task 4 grouped three-fold procedure estimate: 0.9544743099428752
Task 4 prediction counts: {0: 4748, 1: 6024, 2: 3572, 3: 6641, 4: 4904}


## Save and verify the five-task CSV

The new file is a local Kaggle draft. Its JSON card records the exact source artifacts and SHA-256. Task 5 remains the previous unsupervised candidate; its rank-stability proxy is not macro F₃.

In [4]:
output = base.copy()
output["task3"] = t3_pred
output["task4"] = t4_by_id.task4.to_numpy(dtype=int)
assert output.columns.tolist() == columns and output.id.tolist() == sample.id.tolist()
assert output.notna().all().all()
for task in (1,2,4,5):
    assert np.isfinite(output[f"task{task}"].to_numpy(dtype=float)).all()
assert (output.task3 >= 0).all()
assert set(output.task4) <= set(range(5))
assert output[["task1","task2","task5"]].equals(base[["task1","task2","task5"]])
assert (output.task3 != base.task3).any() and (output.task4 != base.task4).any()
out_path = SRC / "submission" / f"{NAME}.csv"
out_card_path = SRC / "submission" / f"{NAME}.json"
if not (out_path.exists() and out_card_path.exists()):
    assert not (out_path.exists() or out_card_path.exists()), 'Inspect incomplete submission artifact'
    output.to_csv(out_path, index=False)
    sources = dict(base_card["sources"])
    sources["task3"] = {"model_name": t3_name,
        "model_card_path": str(t3_card_path.relative_to(SRC)).replace('\\','/'),
        "model_path": t3_card["model_path"], "data_name": t3_card["data_name"],
        "data_card_path": t3_card["data_card_path"],
        "test_path": t3_data["splits"]["test"]["path"],
        "evaluation_metric": t3_eval["metric"], "evaluation_score": t3_eval["score"]}
    t4_model_card = json.loads((SRC / "task4" / "save" / f"{t4_name}.json").read_text(encoding="utf-8"))
    sources["task4"] = {"model_name": t4_name,
        "model_card_path": str((SRC / "task4" / "save" / f"{t4_name}.json").relative_to(SRC)).replace('\\','/'),
        "model_path": t4_model_card["model_path"], "data_name": t4_model_card["data_name"],
        "data_card_path": t4_model_card["data_card_path"],
        "test_path": t4_part_card["submission_path"],
        "evaluation_metric": t4_eval["metric"], "evaluation_score": t4_eval["score"]}
    detail = {"name": NAME, "created_at": datetime.now(timezone.utc).isoformat(),
        "card": "Task 3 zero-aware blend and Task 4 geometry ResNet18; Tasks 1/2/5 copied from verified v5",
        "submission_path": f"submission/{NAME}.csv", "format": "comma-separated CSV",
        "rows": len(output), "columns": columns,
        "sha256": hashlib.sha256(out_path.read_bytes()).hexdigest(),
        "base_submission": BASE_NAME, "base_sha256": base_card["sha256"],
        "changed_tasks": [3,4], "sources": sources,
        "task3_id_mapping": "unchanged Test row position, as in v5 assembly",
        "task4_id_mapping": "current ANS IDs joined by ID",
        "task5_metric_note": "rank stability is an unlabeled proxy, not macro F3"}
    out_card_path.write_text(json.dumps(detail, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
saved = pd.read_csv(out_path)
saved_card = json.loads(out_card_path.read_text(encoding="utf-8"))
assert saved.equals(output) and saved.id.tolist() == sample.id.tolist()
assert saved_card['sha256'] == hashlib.sha256(out_path.read_bytes()).hexdigest()
print('Saved', out_path, 'rows', len(saved), 'sha256', saved_card['sha256'])

Saved

 C:\Users\BTCOM\Desktop\ml-final\src\submission\kaggle_t12345_gate_resnet_v6.csv rows 25889 sha256 08aaf747ed08161f99049e561bb6d1229e68a3de4fd1a0afea79b3aeee1a632e


## Optional Task 5 label inversion draft

Task 5 has no supplied labels, so the anomaly model’s binary orientation cannot be checked offline. This separate file flips only its 0/1 output for a controlled Kaggle comparison. The original five-task submission remains unchanged. The user later confirmed that v6 has the correct Task 5 label mapping; keep v6 for future submissions. The flipped v7 file remains a comparison artifact. No numeric Task 5 F3 score was supplied.

In [5]:
FLIP_NAME = "kaggle_t12345_gate_resnet_t5flip_v7"
flip_path = SRC / "submission" / f"{FLIP_NAME}.csv"
flip_card_path = SRC / "submission" / f"{FLIP_NAME}.json"
source = pd.read_csv(SRC / "submission" / f"{NAME}.csv")
source_card = json.loads((SRC / "submission" / f"{NAME}.json").read_text(encoding="utf-8"))
assert hashlib.sha256((SRC / source_card["submission_path"]).read_bytes()).hexdigest() == source_card["sha256"]
assert source.columns.tolist() == columns and source.id.tolist() == sample.id.tolist()
assert set(source.task5) == {0, 1}
flipped = source.copy()
flipped["task5"] = 1 - flipped["task5"].astype(int)
assert flipped[["id", "task1", "task2", "task3", "task4"]].equals(
    source[["id", "task1", "task2", "task3", "task4"]])
assert (flipped.task5.to_numpy() == 1 - source.task5.to_numpy()).all()
assert flipped.notna().all().all() and flipped.id.is_unique
if not (flip_path.exists() and flip_card_path.exists()):
    assert not (flip_path.exists() or flip_card_path.exists()), "Inspect incomplete flipped artifact"
    flipped.to_csv(flip_path, index=False)
    flipped_sources = dict(source_card["sources"])
    flipped_sources["task5"] = dict(flipped_sources["task5"])
    flipped_sources["task5"]["label_transform"] = "1 - original Task 5 prediction"
    flipped_sources["task5"]["metric_note"] = "Unlabeled alternative; proxy score does not assess label orientation"
    detail = {"name": FLIP_NAME, "created_at": datetime.now(timezone.utc).isoformat(),
        "card": "Task 5 binary label inversion of verified v6; Tasks 1-4 unchanged",
        "submission_path": f"submission/{FLIP_NAME}.csv", "format": "comma-separated CSV",
        "rows": len(flipped), "columns": columns,
        "sha256": hashlib.sha256(flip_path.read_bytes()).hexdigest(),
        "base_submission": NAME, "base_sha256": source_card["sha256"],
        "changed_tasks": [5], "task5_transform": "1 - original Task 5 prediction",
        "sources": flipped_sources,
        "task5_metric_note": "No offline labels; flipping 0/1 has no measured F3"}
    flip_card_path.write_text(json.dumps(detail, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
saved_flip = pd.read_csv(flip_path)
saved_card = json.loads(flip_card_path.read_text(encoding="utf-8"))
assert saved_flip.equals(flipped)
assert saved_card["sha256"] == hashlib.sha256(flip_path.read_bytes()).hexdigest()
print("Saved", flip_path, "rows", len(saved_flip), "Task 5 counts", saved_flip.task5.value_counts().sort_index().to_dict())

Saved C:\Users\BTCOM\Desktop\ml-final\src\submission\kaggle_t12345_gate_resnet_t5flip_v7.csv rows 25889 Task 5 counts {0: 1365, 1: 24524}
